In [1]:
import duckdb

### 1.First load and query directly

In [2]:
duckdb.sql("SELECT 42 AS answer, 'hello' AS greeting")

┌────────┬──────────┐
│ answer │ greeting │
│ int32  │ varchar  │
├────────┼──────────┤
│     42 │ hello    │
└────────┴──────────┘

In [3]:
duckdb.sql("SELECT * FROM 'data/dim_country.csv'")

┌────────────┬──────────────┬─────────┐
│ country_id │ country_name │ region  │
│   int64    │   varchar    │ varchar │
├────────────┼──────────────┼─────────┤
│          1 │ France       │ EU      │
│          2 │ Germany      │ EU      │
│          3 │ Spain        │ EU      │
│          4 │ USA          │ NA      │
│          5 │ Canada       │ NA      │
└────────────┴──────────────┴─────────┘

In [4]:
duckdb.sql("DESCRIBE SELECT * FROM 'data/fact_order_items.csv'")

┌───────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name  │ column_type │  null   │   key   │ default │  extra  │
│    varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ order_item_id │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ order_id      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ product_id    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ quantity      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ unit_price    │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ discount_pct  │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
└───────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [5]:
duckdb.sql("DESCRIBE SELECT * FROM 'data/fact_orders.csv'")

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ order_id    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_id │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ order_ts    │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ status      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [6]:
duckdb.sql("SELECT count(*) AS n, count(discount_pct)" 
            "AS non_null FROM 'data/fact_order_items.csv'")

┌───────┬──────────┐
│   n   │ non_null │
│ int64 │  int64   │
├───────┼──────────┤
│  2643 │     2377 │
└───────┴──────────┘

In [7]:
duckdb.sql("SELECT email IS NULL AS is_null, count(*) "
           "FROM 'data/dim_customer.csv' GROUP BY 1")

┌─────────┬──────────────┐
│ is_null │ count_star() │
│ boolean │    int64     │
├─────────┼──────────────┤
│ false   │          190 │
│ true    │           10 │
└─────────┴──────────────┘

In [8]:
r = duckdb.sql("SELECT status, count(*) AS n "
               "FROM 'data/fact_orders.csv' GROUP BY status ORDER BY n DESC")

r                # pretty table (in a REPL / notebook)
# r.fetchall()     # list of Python tuples: [('completed', 1051), ('shipped', 230), ...]


┌───────────┬───────┐
│  status   │   n   │
│  varchar  │ int64 │
├───────────┼───────┤
│ completed │  1051 │
│ shipped   │   230 │
│ cancelled │   150 │
│ refunded  │    69 │
└───────────┴───────┘

In [9]:
r = duckdb.sql("SELECT * FROM 'data/fact_orders.csv' LIMIT 5")
row0 = r.fetchall()[0]
print(f"First line : {row0}")
print("======DataFrame======:")
print(r.df())


First line : (1, 40, datetime.datetime(2025, 5, 28, 14, 51, 36), 'shipped')
======DataFrame======:
   order_id  customer_id            order_ts     status
0         1           40 2025-05-28 14:51:36    shipped
1         2          161 2025-03-03 09:18:50  completed
2         3          116 2025-01-08 07:15:44  completed
3         4           27 2025-06-07 11:22:45  cancelled
4         5          157 2025-01-31 19:48:39  completed


In [10]:
print(row0[2].isoformat())
print(row0)

2025-05-28T14:51:36
(1, 40, datetime.datetime(2025, 5, 28, 14, 51, 36), 'shipped')


In [11]:
duckdb.sql("DESCRIBE SELECT * FROM 'data/dim_customer.csv'")

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ customer_id │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ name        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ email       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ city_id     │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ signup_date │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [12]:
duckdb.sql("DESCRIBE SELECT * FROM 'data/payments.jsonl'")

┌─────────────┬──────────────────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │             column_type              │  null   │   key   │ default │  extra  │
│   varchar   │               varchar                │ varchar │ varchar │ varchar │ varchar │
├─────────────┼──────────────────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ order_id    │ BIGINT                               │ YES     │ NULL    │ NULL    │ NULL    │
│ paid_at     │ TIMESTAMP                            │ YES     │ NULL    │ NULL    │ NULL    │
│ method      │ VARCHAR                              │ YES     │ NULL    │ NULL    │ NULL    │
│ amount      │ DOUBLE                               │ YES     │ NULL    │ NULL    │ NULL    │
│ card        │ STRUCT(brand VARCHAR, last4 VARCHAR) │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴──────────────────────────────────────┴─────────┴─────────┴─────────┴─────────┘

### 2. Connect with duckDB

In [38]:
con = duckdb.connect()  # in-memory, disappears when the kernel stops
# con = duckdb.connect('db/analytics.duckdb') # or a file: tables persist between sessions
con.sql("INSTALL sqlite; LOAD sqlite;")

In [39]:
con.sql("ATTACH 'db/marketplace.db' AS m (TYPE sqlite, READ_ONLY)")

In [15]:
con.sql("SHOW ALL TABLES")

┌──────────┬─────────┬──────────────────┬───────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────┬───────────┐
│ database │ schema  │       name       │                               column_names                                │                     column_types                     │ temporary │
│ varchar  │ varchar │     varchar      │                                 varchar[]                                 │                      varchar[]                       │  boolean  │
├──────────┼─────────┼──────────────────┼───────────────────────────────────────────────────────────────────────────┼──────────────────────────────────────────────────────┼───────────┤
│ m        │ main    │ dim_category     │ [category_id, category_name, parent_category_id]                          │ [BIGINT, VARCHAR, BIGINT]                            │ false     │
│ m        │ main    │ dim_city         │ [city_id, city_name, country_id] 

In [16]:
con.sql("DESCRIBE m.fact_orders")

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ order_id    │ BIGINT      │ YES     │ PRI     │ NULL    │ NULL    │
│ customer_id │ BIGINT      │ NO      │ NULL    │ NULL    │ NULL    │
│ order_ts    │ VARCHAR     │ NO      │ NULL    │ NULL    │ NULL    │
│ status      │ VARCHAR     │ NO      │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [17]:
con.sql("""
SELECT customer_id, order_id, order_ts
FROM m.fact_orders
QUALIFY row_number() OVER (PARTITION BY customer_id ORDER BY order_ts DESC) = 1
ORDER BY customer_id
LIMIT 3
""")


┌─────────────┬──────────┬─────────────────────┐
│ customer_id │ order_id │      order_ts       │
│    int64    │  int64   │       varchar       │
├─────────────┼──────────┼─────────────────────┤
│           1 │     1174 │ 2025-06-23 09:32:10 │
│           2 │     1419 │ 2025-05-15 22:07:56 │
│           3 │      391 │ 2025-06-26 15:55:28 │
└─────────────┴──────────┴─────────────────────┘

In [18]:
con.sql("""
SELECT date_trunc('month', order_ts::TIMESTAMP) AS month, status, count(*) AS n
FROM m.fact_orders
GROUP BY ALL
ORDER BY month, status
LIMIT 10
""")


┌─────────────────────┬───────────┬───────┐
│        month        │  status   │   n   │
│      timestamp      │  varchar  │ int64 │
├─────────────────────┼───────────┼───────┤
│ 2025-01-01 00:00:00 │ cancelled │    26 │
│ 2025-01-01 00:00:00 │ completed │   174 │
│ 2025-01-01 00:00:00 │ refunded  │    13 │
│ 2025-01-01 00:00:00 │ shipped   │    36 │
│ 2025-02-01 00:00:00 │ cancelled │    20 │
│ 2025-02-01 00:00:00 │ completed │   163 │
│ 2025-02-01 00:00:00 │ refunded  │    13 │
│ 2025-02-01 00:00:00 │ shipped   │    40 │
│ 2025-03-01 00:00:00 │ cancelled │    20 │
│ 2025-03-01 00:00:00 │ completed │   191 │
└─────────────────────┴───────────┴───────┘
  10 rows                       3 columns

In [19]:
con.sql("""
SELECT c.customer_id, d.city_name, e.country_name FROM m.dim_customer AS c
JOIN m.dim_city AS d ON c.city_id = d.city_id
JOIN m.dim_country AS e ON d.country_id = e.country_id
ORDER BY c.customer_id LIMIT 10
""")

┌─────────────┬───────────┬──────────────┐
│ customer_id │ city_name │ country_name │
│    int64    │  varchar  │   varchar    │
├─────────────┼───────────┼──────────────┤
│           1 │ Munich    │ Germany      │
│           2 │ Toronto   │ Canada       │
│           3 │ Munich    │ Germany      │
│           4 │ Munich    │ Germany      │
│           5 │ Montreal  │ Canada       │
│           6 │ Barcelona │ Spain        │
│           7 │ Barcelona │ Spain        │
│           8 │ Barcelona │ Spain        │
│           9 │ Toronto   │ Canada       │
│          10 │ Montreal  │ Canada       │
└─────────────┴───────────┴──────────────┘
  10 rows                      3 columns

In [20]:
con.sql("""
SELECT e.country_name, count(*) FROM m.dim_customer AS c
JOIN m.dim_city AS d ON c.city_id = d.city_id
JOIN m.dim_country AS e ON d.country_id = e.country_id
GROUP BY e.country_name
HAVING count(*) >= 40
""")

┌──────────────┬──────────────┐
│ country_name │ count_star() │
│   varchar    │    int64     │
├──────────────┼──────────────┤
│ USA          │           45 │
│ Canada       │           49 │
└──────────────┴──────────────┘

In [21]:
con.sql("SELECT * EXCLUDE (order_ts) FROM m.fact_orders LIMIT 2")

┌──────────┬─────────────┬───────────┐
│ order_id │ customer_id │  status   │
│  int64   │    int64    │  varchar  │
├──────────┼─────────────┼───────────┤
│        1 │          40 │ shipped   │
│        2 │         161 │ completed │
└──────────┴─────────────┴───────────┘

### 3 Persist to a DuckDB file

In [22]:
con2 = duckdb.connect('db/analytics.duckdb')
con2.sql("CREATE TABLE orders AS SELECT * FROM 'data/fact_orders.csv'")
con2.sql("SELECT count(*) FROM orders")
con2.close()

In [28]:
con2 = duckdb.connect('db/analytics.duckdb')

In [29]:
con2.sql("""
SELECT * FROM orders LIMIT 10
""")

┌──────────┬─────────────┬─────────────────────┬───────────┐
│ order_id │ customer_id │      order_ts       │  status   │
│  int64   │    int64    │      timestamp      │  varchar  │
├──────────┼─────────────┼─────────────────────┼───────────┤
│        1 │          40 │ 2025-05-28 14:51:36 │ shipped   │
│        2 │         161 │ 2025-03-03 09:18:50 │ completed │
│        3 │         116 │ 2025-01-08 07:15:44 │ completed │
│        4 │          27 │ 2025-06-07 11:22:45 │ cancelled │
│        5 │         157 │ 2025-01-31 19:48:39 │ completed │
│        6 │         150 │ 2025-05-09 20:39:23 │ completed │
│        7 │          85 │ 2025-06-07 21:42:56 │ completed │
│        8 │          36 │ 2025-01-31 02:05:48 │ completed │
│        9 │          94 │ 2025-05-12 08:04:46 │ shipped   │
│       10 │          65 │ 2025-01-11 03:10:31 │ completed │
└──────────┴─────────────┴─────────────────────┴───────────┘
  10 rows                                        4 columns

In [30]:
con2.sql("SHOW TABLES")

┌─────────┐
│  name   │
│ varchar │
├─────────┤
│ orders  │
└─────────┘

In [31]:
con2.sql("""
    SELECT order_id, order_ts,
       LAG(order_ts) OVER (PARTITION BY customer_id ORDER BY order_ts) AS prev_ts,
       round(julian(order_ts)
             - julian(LAG(order_ts) OVER (PARTITION BY customer_id ORDER BY order_ts)), 1) AS days_since_prev
FROM orders WHERE customer_id = 1 ORDER BY order_ts
""")

┌──────────┬─────────────────────┬─────────────────────┬─────────────────┐
│ order_id │      order_ts       │       prev_ts       │ days_since_prev │
│  int64   │      timestamp      │      timestamp      │     double      │
├──────────┼─────────────────────┼─────────────────────┼─────────────────┤
│      360 │ 2025-01-30 21:48:13 │ NULL                │            NULL │
│      101 │ 2025-02-11 10:36:53 │ 2025-01-30 21:48:13 │            11.5 │
│     1438 │ 2025-03-13 10:22:20 │ 2025-02-11 10:36:53 │            30.0 │
│      450 │ 2025-04-29 05:09:06 │ 2025-03-13 10:22:20 │            46.8 │
│     1195 │ 2025-05-13 11:11:53 │ 2025-04-29 05:09:06 │            14.3 │
│      888 │ 2025-06-15 13:47:00 │ 2025-05-13 11:11:53 │            33.1 │
│      571 │ 2025-06-17 07:52:41 │ 2025-06-15 13:47:00 │             1.8 │
│     1342 │ 2025-06-20 14:48:07 │ 2025-06-17 07:52:41 │             3.3 │
│     1174 │ 2025-06-23 09:32:10 │ 2025-06-20 14:48:07 │             2.8 │
└──────────┴─────────────

In [37]:
con2.sql("""
SELECT customer_id, order_id, order_ts
FROM orders
QUALIFY row_number() OVER (PARTITION BY customer_id ORDER BY order_ts DESC) <= 3
ORDER BY customer_id, order_ts
LIMIT 9
""")

┌─────────────┬──────────┬─────────────────────┐
│ customer_id │ order_id │      order_ts       │
│    int64    │  int64   │      timestamp      │
├─────────────┼──────────┼─────────────────────┤
│           1 │      571 │ 2025-06-17 07:52:41 │
│           1 │     1342 │ 2025-06-20 14:48:07 │
│           1 │     1174 │ 2025-06-23 09:32:10 │
│           2 │      115 │ 2025-03-21 14:54:24 │
│           2 │     1467 │ 2025-05-15 08:09:11 │
│           2 │     1419 │ 2025-05-15 22:07:56 │
│           3 │      262 │ 2025-05-04 02:07:40 │
│           3 │      319 │ 2025-06-11 11:57:14 │
│           3 │      391 │ 2025-06-26 15:55:28 │
└─────────────┴──────────┴─────────────────────┘

In [42]:
con3 = duckdb.connect()
con3.sql("ATTACH 'db/marketplace.db' AS m (TYPE sqlite)")
con3.sql("SELECT * FROM m.fact_orders LIMIT 10")

┌──────────┬─────────────┬─────────────────────┬───────────┐
│ order_id │ customer_id │      order_ts       │  status   │
│  int64   │    int64    │       varchar       │  varchar  │
├──────────┼─────────────┼─────────────────────┼───────────┤
│        1 │          40 │ 2025-05-28 14:51:36 │ shipped   │
│        2 │         161 │ 2025-03-03 09:18:50 │ completed │
│        3 │         116 │ 2025-01-08 07:15:44 │ completed │
│        4 │          27 │ 2025-06-07 11:22:45 │ cancelled │
│        5 │         157 │ 2025-01-31 19:48:39 │ completed │
│        6 │         150 │ 2025-05-09 20:39:23 │ completed │
│        7 │          85 │ 2025-06-07 21:42:56 │ completed │
│        8 │          36 │ 2025-01-31 02:05:48 │ completed │
│        9 │          94 │ 2025-05-12 08:04:46 │ shipped   │
│       10 │          65 │ 2025-01-11 03:10:31 │ completed │
└──────────┴─────────────┴─────────────────────┴───────────┘
  10 rows                                        4 columns

In [43]:
con.close()
con2.close()
con3.close()

### 4. Anlytics and connection with pandas

In [48]:
con = duckdb.connect("db/analytics.duckdb")
con.sql("ATTACH 'db/marketplace.db' AS m (TYPE sqlite, READ_ONLY)")

In [58]:
items  = con.sql("SELECT * FROM m.fact_order_items").df()
orders = con.sql("SELECT * FROM 'data/fact_orders.csv'").df()
orders.dtypes

order_id                int64
customer_id             int64
order_ts       datetime64[us]
status                 object
dtype: object

In [59]:
con.sql("SELECT status, count(*) AS n FROM orders GROUP BY status ORDER BY n DESC")

┌───────────┬───────┐
│  status   │   n   │
│  varchar  │ int64 │
├───────────┼───────┤
│ completed │  1051 │
│ shipped   │   230 │
│ cancelled │   150 │
│ refunded  │    69 │
└───────────┴───────┘

In [60]:
recent = orders[orders.order_ts >= '2025-06-01']
con.sql("SELECT count(*) AS n FROM recent").fetchall()

[(239,)]

In [62]:
con.sql("""
SELECT c.name, count(*) AS n
FROM orders JOIN m.dim_customer AS c USING (customer_id)
GROUP BY 1 ORDER BY n DESC, 1 LIMIT 3""")

┌─────────────┬───────┐
│    name     │   n   │
│   varchar   │ int64 │
├─────────────┼───────┤
│ Ana Roy     │    34 │
│ Iris Martin │    34 │
│ Rosa Martin │    33 │
└─────────────┴───────┘

In [68]:
print(orders.iloc[:5,:])
r = con.sql("SELECT status, count(*) n FROM orders GROUP BY status")
r.df()           # pandas DataFrame
# r.fetchall()     # list of tuples
# r.fetchnumpy()   # dict of NumPy arrays
# r.arrow()        # PyArrow (a reader; use .read_all() for a Table)


   order_id  customer_id            order_ts     status
0         1           40 2025-05-28 14:51:36    shipped
1         2          161 2025-03-03 09:18:50  completed
2         3          116 2025-01-08 07:15:44  completed
3         4           27 2025-06-07 11:22:45  cancelled
4         5          157 2025-01-31 19:48:39  completed


,status,n
0,shipped,230
1,refunded,69
2,cancelled,150
3,completed,1051


In [70]:
# SQL
r = con.sql("SELECT status, count(*) AS n "
        "FROM orders GROUP BY status ORDER BY n DESC").df()
print(r)

# pandas
r = orders.groupby('status').size().sort_values(ascending=False)
print(r)


      status     n
0  completed  1051
1    shipped   230
2  cancelled   150
3   refunded    69
status
completed    1051
shipped       230
cancelled     150
refunded       69
dtype: int64


In [81]:
events  = con.sql("SELECT * FROM m.web_events").df()
customers = con.sql("SELECT customer_id, name, city_id FROM m.dim_customer").df()
orders = con.sql("SELECT * FROM m.fact_orders").df()

con.sql("""
WITH ev AS (
    SELECT user_id,
           count(*) AS n_events,
           count(*) FILTER (WHERE event = 'add_to_cart') AS n_carts
    FROM events
    GROUP BY user_id
),
od AS (
    SELECT customer_id, count(*) AS n_orders
    FROM orders
    GROUP BY customer_id
)
SELECT ev.user_id, c.name, ev.n_events, ev.n_carts,
 COALESCE(od.n_orders, 0) AS n_orders
FROM ev
JOIN customers AS c ON c.customer_id  = ev.user_id
LEFT JOIN od ON od.customer_id = ev.user_id
ORDER BY n_orders DESC, ev.user_id
""").df().iloc[[1,2,3,4,5, -2, -1],:]

,user_id,name,n_events,n_carts,n_orders
1,179,Ana Roy,33,8,15
2,96,Mia Roy,33,10,14
3,35,Eva Dubois,40,12,13
4,37,Rosa Petit,31,6,13
5,66,Eva Smith,34,5,13
198,91,Ana Weber,27,5,1
199,180,Farid Lopez,25,7,0


In [82]:
ev = (events.groupby('user_id')
            .agg(n_events=('event_id', 'size'),
                 n_carts=('event', lambda s: (s == 'add_to_cart').sum()))
            .reset_index())

od = orders.groupby('customer_id').size().rename('n_orders').reset_index()

result = (ev.merge(customers[['customer_id', 'name']],           # only the columns you need
                   how='left', left_on='user_id', right_on='customer_id')
            .drop(columns='customer_id')                         # duplicate key from this merge
            .merge(od, how='left', left_on='user_id', right_on='customer_id')
            .drop(columns='customer_id')                         # duplicate key from the second merge
            .fillna({'n_orders': 0}).astype({'n_orders': 'int64'})
            .sort_values(['n_orders', 'user_id'], ascending=[False, True])
            .reset_index(drop=True)
            [['user_id', 'name', 'n_events', 'n_carts', 'n_orders']])   # column order
result


,user_id,name,n_events,n_carts,n_orders
0,64,Kim Bernard,32,7,15
1,179,Ana Roy,33,8,15
2,96,Mia Roy,33,10,14
3,35,Eva Dubois,40,12,13
4,37,Rosa Petit,31,6,13
...,...,...,...,...,...
195,195,Olga Weber,38,11,3
196,200,Kim Weber,31,4,3
197,90,Mia Brown,27,8,2
198,91,Ana Weber,27,5,1


In [83]:
con.close()